# Ticketward: training on Colab (T4)

A thin wrapper around `python -m tw_ml.train` (spec §9.5). The code and configs come from the
repository at a pinned commit; this notebook holds no training logic. Runbook: `docs/training.md`.

**Before "Run all"**

- Runtime, Change runtime type: **T4 GPU**.
- Secrets (key icon): add **`HF_TOKEN`** (write access to your private repos) and allow notebook
  access. It goes into the environment only and is never printed.
- Edit the parameters cell: commit SHA, config, private dataset id and revision.

Colab is interactive only: keep the tab open. After a disconnect, run all cells again; each seed
resumes from its private Hub repo's `last-checkpoint/` (checkpoints every 25 steps). One T4 means
the three seeds run one after another.

In [ ]:
# Parameters: edit these, then "Run all" (docs/training.md).
REPO_URL = "https://github.com/AkhilMittapalli/Ticketward"
GIT_SHA = "<40-hex commit SHA>"  # the commit whose code and configs this run uses
# CONFIG: sft_qwen35_2b, sft_qwen3_1p7b, sft_qwen3_4b_2507 or encoder_modernbert (ml/configs/)
CONFIG = "ml/configs/sft_qwen35_2b.yaml"
SEEDS = (42, 1337, 2026)  # one T4: one seed after another
HF_DATASET_ID = "<hf_user>/ticketward-data"  # private dataset; only train/ and val/ are fetched
HF_DATASET_REVISION = "<40-hex dataset commit SHA>"
EXPECTED_TORCH = "2.11"  # platform.torch of the config
GPU_COUNT = 1
RUN_SMOKE_TEST = True  # R-21 smoke test first (SFT configs only)
ALLOW_UNVERIFIED = False  # True only while base revisions are still <verify> (recorded in runs)
RUN_ROOT = "/content/tw-runs"  # ephemeral: the Hub repos keep checkpoints and run files

In [ ]:
# Clone at the pinned commit and verify it.
import os
import re
import subprocess
import sys

assert re.fullmatch(r"[0-9a-f]{40}", GIT_SHA), "set GIT_SHA to a 40-hex commit SHA"
assert re.fullmatch(r"[0-9a-f]{40}", HF_DATASET_REVISION), "set HF_DATASET_REVISION"
subprocess.run(["git", "clone", "--quiet", REPO_URL, "ticketward"], check=True)
subprocess.run(["git", "-C", "ticketward", "checkout", "--quiet", "--detach", GIT_SHA], check=True)
head = subprocess.run(
    ["git", "-C", "ticketward", "rev-parse", "HEAD"], check=True, capture_output=True, text=True
).stdout.strip()
assert head == GIT_SHA, f"checked out {head}, expected {GIT_SHA}"
os.chdir("ticketward")
print("repository at", head)

In [ ]:
# The platform: Python 3.12/3.13, the platform torch (never installed by us) and T4 GPUs.
import torch

print(sys.version)
assert sys.version_info[:2] in {(3, 12), (3, 13)}, "tw-ml needs Python 3.12 or 3.13"
torch_minor = ".".join(torch.__version__.split("+")[0].split(".")[:2])
assert torch_minor == EXPECTED_TORCH, f"torch {torch.__version__}, config expects {EXPECTED_TORCH}"
assert torch.cuda.is_available(), "select a GPU accelerator"
gpus = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
print(torch.__version__, torch.version.cuda, gpus)
assert len(gpus) == GPU_COUNT, f"expected {GPU_COUNT} GPU(s), got {gpus}"
assert all("T4" in name for name in gpus), f"expected T4 GPUs, got {gpus}"

In [ ]:
# Pinned installs: the hashed export of ml/uv.lock (no torch, triton or CUDA wheels), then tw_ml.
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-deps",
        "--require-hashes",
        "-r",
        "ml/requirements/train.txt",
    ],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "--no-deps", "-e", "./ml"], check=True
)

In [ ]:
# HF_TOKEN from Colab Secrets into the environment only (never printed or logged).
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
assert os.environ["HF_TOKEN"], "add the HF_TOKEN secret and allow notebook access"

In [ ]:
# Private dataset at the pinned revision: train/ and val/ only.
subprocess.run(
    [
        sys.executable,
        "-m",
        "tw_ml.train.hub",
        "fetch-data",
        "--repo",
        HF_DATASET_ID,
        "--revision",
        HF_DATASET_REVISION,
        "--out",
        "data/generated",
    ],
    check=True,
)

In [ ]:
# R-21 smoke test (SFT configs): fp16, fp32 adapters, loss type, memory, time, resume.
FLAGS = ["--allow-unverified"] if ALLOW_UNVERIFIED else []
os.makedirs(RUN_ROOT, exist_ok=True)
if RUN_SMOKE_TEST and os.path.basename(CONFIG).startswith("sft_"):
    subprocess.run(
        [
            sys.executable,
            "-m",
            "tw_ml.train.smoke",
            "--config",
            CONFIG,
            "--steps",
            "20",
            "--run-root",
            RUN_ROOT,
            *FLAGS,
        ],
        check=True,
        env={**os.environ, "CUDA_VISIBLE_DEVICES": "0"},
    )

In [ ]:
# One T4: the seeds run one after another (each resumes from its Hub repo).
def train(seed: int, gpu: int) -> subprocess.Popen[bytes]:
    """Start one seed on one GPU; its log goes to RUN_ROOT/train-s<seed>.log."""
    log = open(f"{RUN_ROOT}/train-s{seed}.log", "wb")  # noqa: SIM115 - the process owns it
    return subprocess.Popen(
        [
            sys.executable,
            "-m",
            "tw_ml.train",
            "--config",
            CONFIG,
            "--seed",
            str(seed),
            "--run-root",
            RUN_ROOT,
            *FLAGS,
        ],
        env={**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)},
        stdout=log,
        stderr=subprocess.STDOUT,
    )


for seed in SEEDS:
    code = train(seed, 0).wait()
    print("seed", seed, "exit code", code)
    assert code == 0, f"see {RUN_ROOT}/train-s{seed}.log"

In [ ]:
# Push each seed's run files (MLflow store, epoch adapters, choice, summaries) again.
for seed in SEEDS:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "tw_ml.train.hub",
            "push-run",
            "--config",
            CONFIG,
            "--seed",
            str(seed),
            "--run-root",
            RUN_ROOT,
        ],
        check=True,
    )